# Lab 2.1 — Streaming on Mantle

**Duration:** ~10 min · **Level:** L200 · **Lab 2 of 4 — part 1/3**

Streaming on Mantle uses **Server-Sent Events (SSE)** on every surface
(`/v1/chat/completions`, `/v1/responses`, `/anthropic/v1/messages`), but the
*event taxonomy* is different on each. This notebook:

1. Streams a completion through Chat Completions and shows how to accumulate
   `delta.content` fragments.
2. Streams through the Responses API and shows the typed events
   (`response.output_text.delta`, `response.completed`).
3. Streams through Anthropic Messages and shows `content_block_delta`.
4. Compares the three side-by-side so you know which one to reach for.


In [ ]:
import os
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "src" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

from src.common.mantle import (
    openai_client, anthropic_client,
    GPT_OSS_120B, CLAUDE_OPUS_47,
)

client = openai_client()
anth = anthropic_client()
print("ready:", os.environ["AWS_REGION"])


## 1. Chat Completions stream

Pass `stream=True`. Each yielded chunk exposes
`chunk.choices[0].delta.content` (string fragment) and, optionally,
`delta.tool_calls` for tool streaming (covered in Lab 2.2).

Set `stream_options={"include_usage": True}` to get a terminal chunk with
`usage` populated — otherwise you don't know how many tokens the stream
actually consumed.


In [ ]:
PROMPT = "Write a 60-word haiku-ish poem about the Nitro hypervisor."

t0 = time.time()
ttft = None
frags = 0
full_text = []
usage = None                          # populated by the terminal chunk

stream = client.chat.completions.create(
    model=GPT_OSS_120B,
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=120,
    stream=True,
    stream_options={"include_usage": True},
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        if ttft is None:
            ttft = time.time() - t0
        piece = chunk.choices[0].delta.content
        full_text.append(piece)
        print(piece, end="", flush=True)
        frags += 1
    if chunk.usage:
        usage = chunk.usage

print("\n\n--- stream stats ---")
print(f"TTFT            : {ttft:.2f} s" if ttft else "TTFT            : n/a (no content)")
print(f"total fragments : {frags}")
if usage:
    print(f"prompt_tokens     = {usage.prompt_tokens}")
    print(f"completion_tokens = {usage.completion_tokens}")
else:
    print("usage chunk not emitted (stream_options.include_usage was ignored?)")


## 2. Responses API stream

The Responses API uses *typed* events instead of raw `delta.content` strings.
Each event has a `type` field; the ones you'll almost always handle are:

- `response.output_text.delta` — incremental text for the visible answer.
- `response.function_call_arguments.delta` / `.done` — tool-call arguments
  (Lab 2.2 covers these).
- `response.completed` — final event, with full `usage` on the nested
  `.response.usage`.

The event tree is richer than Chat Completions but also friendlier for
tool-heavy workflows because you don't have to reassemble arguments
byte-by-byte.


In [ ]:
t0 = time.time()
ttft = None
pieces = []
final_usage = None

stream = client.responses.create(
    model=GPT_OSS_120B,
    input=PROMPT,
    max_output_tokens=120,
    stream=True,
)

for event in stream:
    etype = getattr(event, "type", "")
    if etype == "response.output_text.delta":
        if ttft is None:
            ttft = time.time() - t0
        pieces.append(event.delta)
        print(event.delta, end="", flush=True)
    elif etype == "response.completed":
        final_usage = event.response.usage

print("\n\n--- stream stats ---")
print(f"TTFT          : {ttft:.2f} s" if ttft else "TTFT          : n/a")
if final_usage:
    print(f"input_tokens  : {final_usage.input_tokens}")
    print(f"output_tokens : {final_usage.output_tokens}")
else:
    print("no response.completed event observed")


## 3. Anthropic Messages stream

The Anthropic path uses its native SSE event names: `message_start`,
`content_block_start`, `content_block_delta`, `content_block_stop`,
`message_delta`, `message_stop`. The SDK gives you an iterator of typed
objects — you almost always want the text accumulator, which the SDK
exposes as `.text_stream`.


In [ ]:
t0 = time.time()
ttft = None
pieces = []

with anth.messages.stream(
    model=CLAUDE_OPUS_47,
    max_tokens=200,
    messages=[{"role": "user", "content": PROMPT}],
) as stream:
    for text in stream.text_stream:
        if ttft is None:
            ttft = time.time() - t0
        pieces.append(text)
        print(text, end="", flush=True)
    final = stream.get_final_message()

print("\n\n--- stream stats ---")
print(f"TTFT         : {ttft:.2f} s")
print(f"input_tokens : {final.usage.input_tokens}")
print(f"output_tokens: {final.usage.output_tokens}")
print(f"stop_reason  : {final.stop_reason}")


## 4. Event-taxonomy cheat sheet

| Concept | Chat Completions | Responses API | Anthropic Messages |
|---|---|---|---|
| Start of stream | First chunk with `choices[0].delta.role` | `response.created` | `message_start` |
| Visible text | `delta.content` | `response.output_text.delta` | `content_block_delta` (text) |
| Tool-call name/id | `delta.tool_calls[i].function.name` / `.id` | `response.output_item.added` (function_call) | `content_block_start` (tool_use) |
| Tool-call args | `delta.tool_calls[i].function.arguments` (byte fragments) | `response.function_call_arguments.delta`/`.done` (full JSON on `.done`) | `content_block_delta` with `input_json_delta` |
| Reasoning / thinking | not surfaced | `response.reasoning.delta` / `summary.text.delta` | `content_block_delta` with `thinking_delta` |
| Terminal usage | last chunk iff `stream_options.include_usage=True` | `response.completed.response.usage` | `message_delta` + `message_stop` |

**Takeaway:** if you only need text, all three are roughly equivalent. If
you need tool calls or reasoning tokens in the stream, the Responses API is
the easiest to consume.


## 5. Common pitfalls

- **Forgetting `stream_options`.** Without it, Chat Completions streams do
  *not* include a terminal `usage` block; you'll silently miss
  input/output token counts.
- **Accumulating `tool_calls` wrong.** Each Chat Completions chunk carries
  *only the new arguments fragment* — you must concatenate them keyed by
  `tool_calls[i].index`. Lab 2.2 has a reusable accumulator.
- **Timing out on slow models.** Extended-thinking models (Opus 4.7 with
  `thinking`, Kimi K2 Thinking) can take 15–60 s to produce the first token.
  Bump your HTTP client read timeout accordingly.
- **Buffering in the middle.** Nginx / ALB / some corporate proxies buffer
  SSE by default. If you see "all at once" output instead of incremental,
  disable response buffering in your infra layer.
